[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pythonanywhere/pypath/blob/main/notebooks/module6/02-image-processing.ipynb)

# Module 6.2 — Image Processing
**Module 6: Computer Vision** | Estimated time: 25 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Apply Gaussian, Median, and Bilateral blurs and explain when to use each
- Understand convolution kernels and apply custom filters with `cv2.filter2D`
- Use global thresholding (binary and Otsu) and adaptive thresholding
- Perform morphological operations: erosion, dilation, opening, closing, and gradient
- Detect and draw contours, extract bounding boxes, and measure area and perimeter
- Count objects in an image using contour filtering

In [ ]:
!pip install opencv-python-headless --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests, os

print(f'OpenCV {cv2.__version__}')

def show_grid(images, titles, cols=3, figsize=(15, 5)):
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for ax, img, title in zip(axes, images, titles):
        cmap = 'gray' if len(img.shape) == 2 else None
        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    for ax in axes[len(images):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Download a high-contrast sample (coins-like synthetic for morphology)
os.makedirs('/tmp/cv_images', exist_ok=True)
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e9/Felis_silvestris_silvestris_small_gradual_decrease_of_quality.png/280px-Felis_silvestris_silvestris_small_gradual_decrease_of_quality.png'
resp = requests.get(url)
with open('/tmp/cv_images/cat.png', 'wb') as f:
    f.write(resp.content)

img_bgr = cv2.imread('/tmp/cv_images/cat.png')
if img_bgr is None:
    # Synthetic fallback
    img_bgr = np.random.randint(100, 200, (300, 400, 3), dtype=np.uint8)
    cv2.circle(img_bgr, (100, 100), 50, (200, 50, 50), -1)
    cv2.circle(img_bgr, (250, 150), 40, (50, 200, 50), -1)
    cv2.rectangle(img_bgr, (300, 200), (380, 270), (50, 50, 200), -1)
    print('Using synthetic fallback image')
else:
    print(f'Image loaded: {img_bgr.shape}')

img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

## Filtering and Blurring

Blurring (smoothing) reduces noise and detail. The three most common OpenCV blur methods are:

| Method | Best for | Preserves edges? |
|---|---|---|
| **Gaussian** | General smoothing, Gaussian noise | No |
| **Median** | Salt-and-pepper noise | Somewhat |
| **Bilateral** | Noise removal while keeping edges sharp | Yes |

All three accept a kernel size (`ksize`). Larger kernels = stronger blur.

In [ ]:
# Add salt-and-pepper noise for demonstration
noisy = img_gray.copy().astype(np.float32)
noise = np.random.choice([0, 255, np.nan], size=noisy.shape,
                         p=[0.05, 0.05, 0.90])
noise = np.where(np.isnan(noise), noisy, noise)
noisy_img = noise.astype(np.uint8)

# Apply blurs
blur_gaussian = cv2.GaussianBlur(noisy_img, (7, 7), sigmaX=2)
blur_median   = cv2.medianBlur(noisy_img, 7)
blur_bilateral= cv2.bilateralFilter(noisy_img, d=9,
                                    sigmaColor=75, sigmaSpace=75)

show_grid(
    [noisy_img, blur_gaussian, blur_median, blur_bilateral],
    ['Noisy Original', 'Gaussian (7×7, σ=2)',
     'Median (k=7)', 'Bilateral (d=9)'],
    cols=4, figsize=(16, 4)
)

print('Gaussian: fast general-purpose blur')
print('Median  : excellent for salt-and-pepper noise')
print('Bilateral: preserves edges (but slower)')

## Custom Kernels with cv2.filter2D

A **convolution kernel** is a small matrix that slides over the image. Each output pixel is the weighted sum of its neighbourhood. This underpins blurring, sharpening, and edge detection.

Common kernels:
- **Box blur**: all 1/9 — simple average
- **Sharpen**: centre weight +5, neighbours −1 — amplifies differences
- **Emboss**: diagonal gradient — gives a 3-D raised effect

In [ ]:
img_work = cv2.GaussianBlur(img_gray, (3, 3), 0)  # start with slight blur

# Box blur kernel (3×3 average)
kernel_box = np.ones((5, 5), np.float32) / 25

# Sharpen kernel
kernel_sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)

# Emboss kernel
kernel_emboss = np.array([
    [-2, -1, 0],
    [-1,  1, 1],
    [ 0,  1, 2]
], dtype=np.float32)

# Edge detection (Laplacian-like)
kernel_edge = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

results = {
    'Box Blur (5×5)':   cv2.filter2D(img_work, -1, kernel_box),
    'Sharpen':          cv2.filter2D(img_work, -1, kernel_sharpen),
    'Emboss':           cv2.filter2D(img_work, -1, kernel_emboss),
    'Edge Enhance':     cv2.filter2D(img_work, -1, kernel_edge),
}

show_grid([img_work] + list(results.values()),
          ['Original Grayscale'] + list(results.keys()),
          cols=5, figsize=(18, 4))

print('filter2D applies any custom kernel to an image.')
print('Kernel values must sum to 1 for brightness-preserving filters.')

## Thresholding

Thresholding converts a grayscale image to binary (black and white) by comparing each pixel to a threshold value.

- **Binary**: `pixel > threshold → 255`, else `0`
- **Otsu's method**: automatically finds the optimal threshold by minimising intra-class variance
- **Adaptive**: threshold varies per region — useful when lighting is uneven

In [ ]:
img_th = img_gray.copy()

# 1. Simple binary threshold (manual T=127)
_, thresh_binary = cv2.threshold(img_th, 127, 255, cv2.THRESH_BINARY)

# 2. Otsu's method (T chosen automatically)
otsu_t, thresh_otsu = cv2.threshold(
    img_th, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"Otsu's optimal threshold: {otsu_t:.1f}")

# 3. Adaptive Mean
thresh_adapt_mean = cv2.adaptiveThreshold(
    img_th, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    blockSize=21, C=5
)

# 4. Adaptive Gaussian
thresh_adapt_gauss = cv2.adaptiveThreshold(
    img_th, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    blockSize=21, C=5
)

show_grid(
    [img_th, thresh_binary, thresh_otsu,
     thresh_adapt_mean, thresh_adapt_gauss],
    ['Grayscale', 'Binary (T=127)', f"Otsu (T={otsu_t:.0f})",
     'Adaptive Mean', 'Adaptive Gaussian'],
    cols=5, figsize=(18, 4)
)

print('\nWhen to use:')
print('  Binary     → uniform, well-lit images')
print("  Otsu       → bimodal histograms (two clear peaks)")
print('  Adaptive   → uneven lighting, scanned documents')

## Morphological Operations

Morphological operations work on binary images using a structuring element (kernel):

| Operation | Effect |
|---|---|
| **Erosion** | Shrinks white regions; removes small noise |
| **Dilation** | Expands white regions; fills small gaps |
| **Opening** (erode→dilate) | Removes small white blobs |
| **Closing** (dilate→erode) | Fills small holes inside white regions |
| **Gradient** (dilate−erode) | Extracts object outlines |

In [ ]:
# Use the Otsu threshold as our binary image
binary = thresh_otsu.copy()

# Define structuring element
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

eroded   = cv2.erode(binary, kernel, iterations=2)
dilated  = cv2.dilate(binary, kernel, iterations=2)
opened   = cv2.morphologyEx(binary, cv2.MORPH_OPEN,    kernel)
closed   = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,   kernel)
gradient = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel)
tophat   = cv2.morphologyEx(binary, cv2.MORPH_TOPHAT,  kernel)

show_grid(
    [binary, eroded, dilated, opened, closed, gradient],
    ['Binary', 'Eroded', 'Dilated', 'Opening', 'Closing', 'Gradient'],
    cols=6, figsize=(18, 4)
)

print('Practical use:')
print('  Opening  → remove tiny speckles before contour detection')
print('  Closing  → connect broken text strokes')
print('  Gradient → extract precise edges from binary images')

## Contour Detection

Contours are curves that join continuous points along a boundary with the same colour or intensity. `cv2.findContours()` returns a list of contours, each as an array of (x, y) points.

Useful contour functions:
- `cv2.contourArea(cnt)` — area in pixels
- `cv2.arcLength(cnt, closed)` — perimeter
- `cv2.boundingRect(cnt)` — axis-aligned bounding box
- `cv2.minEnclosingCircle(cnt)` — smallest enclosing circle

In [ ]:
# Clean binary image for contour demo
clean = cv2.morphologyEx(thresh_otsu, cv2.MORPH_OPEN,
                         cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))

# Find contours
contours, hierarchy = cv2.findContours(
    clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f'Total contours found: {len(contours)}')

# Filter by area (ignore tiny noise)
min_area = 200
large_cnts = [c for c in contours if cv2.contourArea(c) > min_area]
print(f'Contours with area > {min_area}px: {len(large_cnts)}')

# Draw on colour copy
canvas = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)

for i, cnt in enumerate(large_cnts):
    area  = cv2.contourArea(cnt)
    peri  = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)

    # Draw contour
    cv2.drawContours(canvas, [cnt], -1, (0, 255, 0), 2)
    # Bounding box
    cv2.rectangle(canvas, (x, y), (x + w, y + h), (255, 0, 0), 1)
    # Area label
    cv2.putText(canvas, f'{area:.0f}',
                (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX,
                0.4, (0, 255, 255), 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.imshow(clean, cmap='gray'); plt.title('Clean Binary'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title(f'Contours ({len(large_cnts)} objects)'); plt.axis('off')
plt.tight_layout(); plt.show()

print('\nTop 5 contours by area:')
for c in sorted(large_cnts, key=cv2.contourArea, reverse=True)[:5]:
    print(f'  Area={cv2.contourArea(c):.1f}  Perimeter={cv2.arcLength(c, True):.1f}')

## Practical: Count Objects in an Image

Combining what we have learned: blur → threshold → morphology → contours → count.

In [ ]:
def count_objects(image_bgr, min_area=500, max_area=50000):
    """
    Count distinct objects in a BGR image.
    Returns (count, annotated_image).
    """
    gray    = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, binary = cv2.threshold(blurred, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
    contours, _ = cv2.findContours(
        cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = [c for c in contours
             if min_area < cv2.contourArea(c) < max_area]

    annotated = image_bgr.copy()
    for i, cnt in enumerate(valid):
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.drawContours(annotated, [cnt], -1, (0, 255, 0), 2)
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (255, 128, 0), 2)
        cv2.putText(annotated, str(i + 1), (x + 5, y + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return len(valid), annotated

# Test on our image
count, annotated = count_objects(img_bgr, min_area=300)
print(f'Objects detected: {count}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Detected Objects: {count}'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Summary

| Topic | Key functions |
|---|---|
| Blurring | `cv2.GaussianBlur`, `cv2.medianBlur`, `cv2.bilateralFilter` |
| Custom kernels | `cv2.filter2D(img, -1, kernel)` |
| Threshold | `cv2.threshold`, `cv2.adaptiveThreshold` |
| Morphology | `cv2.erode`, `cv2.dilate`, `cv2.morphologyEx` |
| Contours | `cv2.findContours`, `cv2.drawContours`, `cv2.boundingRect` |

## Practice Exercises

**Exercise 1 — Noise Comparison:**  
Add Gaussian noise (`np.random.normal`) to an image, then apply all three blur types. Calculate the PSNR (Peak Signal-to-Noise Ratio) between each denoised result and the original using `cv2.PSNR()`. Which blur gives the best result for Gaussian noise?

**Exercise 2 — Document Binarisation:**  
Download a photo of handwritten text or a receipt. Compare Otsu's threshold vs. adaptive Gaussian thresholding. Which preserves text better under uneven illumination? Adjust the `blockSize` and `C` parameters to improve readability.

**Exercise 3 — Coin Counter:**  
Create a synthetic image with 10–15 white filled circles of varying sizes on a dark background using `cv2.circle`. Then use the `count_objects()` function to count them. Adjust `min_area` and `max_area` until you get an exact count.